In [64]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import bayesflow as bf

import sys
sys.path.append("../")

from src.generative_models import *
from src.priors import pt_prior

np.set_printoptions(suppress=True)

In [52]:
data = pd.read_csv("../data/pilot_data_prepped.csv")
NUM_SUBJECTS = len(data.id.unique())
SUBJECTS = data.id.unique()
NUM_OBS = 100
NUM_SAMPLES = 1000

In [53]:
emp_data = np.zeros((NUM_SUBJECTS, 2, NUM_OBS, 7))
for i, subject in enumerate(SUBJECTS):
    subject_data_old = data[(data.id == subject) & (data.condition == "old")]
    subject_data_new = data[(data.id == subject) & (data.condition == "new")]
    
    resp_old = subject_data_old.resp.to_numpy()[:, None]
    resp_new = subject_data_new.resp.to_numpy()[:, None]

    context_old = subject_data_old.iloc[:, 8:14].to_numpy() / 50
    context_new = subject_data_new.iloc[:, 8:14].to_numpy() / 50
    emp_data[i, 0] = np.c_[resp_old, context_old]
    emp_data[i, 1] = np.c_[resp_new, context_new]

## Model Fitting

In [54]:
summary_net = bf.networks.SetTransformer(input_dim=7, summary_dim=32)

inference_net = bf.networks.InvertibleNetwork(
    num_params=len(pt_prior.param_names),
    coupling_settings={"dense_args": dict(kernel_regularizer=None), "dropout": False},
)

In [55]:
pt_amortizer = bf.amortizers.AmortizedPosterior(inference_net, summary_net)

pt_trainer = bf.trainers.Trainer(
    generative_model=pt_model, 
    amortizer=pt_amortizer, 
    configurator=pt_configurator, 
    checkpoint_path=f"../checkpoints/pt_model_final",
    max_to_keep=1
)

INFO:root:Loaded loss history from ../checkpoints/pt_model_final/history_100.pkl.
INFO:root:Networks loaded from ../checkpoints/pt_model_final/ckpt-100
INFO:root:Performing a consistency check with provided components...
INFO:root:Done.


In [59]:
pt_post_z = np.zeros((NUM_SUBJECTS, 2, NUM_SAMPLES, 3))

In [58]:
pt_post_z[:, 0].shape

(12, 100, 1000, 3)

In [ ]:
pt_post_z[:, 0] = pt_amortizer.sample(
    {"summary_conditions": emp_data[:, 0].astype(np.float32)},
    NUM_SAMPLES
)
pt_post_z[:, 1] = pt_amortizer.sample(
    {"summary_conditions": emp_data[:, 1].astype(np.float32)},
    NUM_SAMPLES
)

In [65]:
pt_post = pt_post_z * PT_PRIOR_STD + PT_PRIOR_MEAN

In [73]:
pt_post_median = np.mean(pt_post, axis=2)

In [75]:
pt_post_median_mean = np.mean(pt_post_median, axis=0)
pt_post_median_std = np.std(pt_post_median, axis=0)
pt_post_median_mean

array([[1.88137069, 0.80019603, 0.45482955],
       [1.6176872 , 0.59063731, 1.18064853]])

## Posterior Re-simulation